# Lesson 20 Lab — FP8, FP4, NVFP4, and Hardware Boundaries

**Puzzle:** Does Blackwell hardware support mean every framework build exposes the same FP8 or NVFP4 path?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

Keep four layers distinct: numerical format, hardware instruction, library recipe, and framework/operator API. `torch.float8_*` existing does not alone prove an FP8 GEMM path.

### Core mechanism

E4M3 favors precision with less range; E5M2 favors range. Scaled FP8 matmul applies explicit scale factors. Blackwell-specific MXFP8/NVFP4 add block-scale structure and require matching recipes and kernels.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "20-fp8-fp4-nvfp4"
device = require_cuda()
torch.manual_seed(2026 + 20)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Smaller formats reduce traffic and raise theoretical throughput but add scale selection, saturation risk, metadata, and software compatibility constraints.

### What this code tests

The lab calls PyTorch scaled FP8 matmul when available and leaves Transformer Engine/NVFP4 unmeasured rather than equating hardware generation with framework support.

**Experiment:** Attempt native PyTorch FP8 GEMM on the RTX GPU, record error and timing when supported, and separately probe Transformer Engine and NVFP4 APIs.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
import importlib.util
n=1024; a=torch.randn(n,n,device=device,dtype=torch.bfloat16); b=torch.randn(n,n,device=device,dtype=torch.bfloat16); ref=(a@b).float()
probe={"torch_float8_dtype":hasattr(torch,"float8_e4m3fn"),"transformer_engine_installed":importlib.util.find_spec("transformer_engine") is not None}
if probe["torch_float8_dtype"]:
    try:
        a8=a.to(torch.float8_e4m3fn); b8=b.to(torch.float8_e4m3fn)
        one=torch.tensor(1.0,device=device)
        def fp8_scaled_mm():
            return torch._scaled_mm(a8,b8,scale_a=one,scale_b=one,out_dtype=torch.bfloat16)
        out=fp8_scaled_mm().float()
        probe.update({"fp8_gemm":"success","api":"torch._scaled_mm","fp8_error":error_metrics(ref,out),
                      "fp8_timing":cuda_benchmark(fp8_scaled_mm,warmup=5,repeats=15)})
    except Exception as exc: probe.update({"fp8_gemm":"failed","error_type":type(exc).__name__,"error_message":str(exc)[:240]})
probe["nvfp4_backend"]="not_measured"
result=base_result(20,"pytorch-gpu" if probe.get("fp8_gemm")=="success" else "compatibility-probe"); result.update({"probe":probe,
    "conclusion":"Framework-level FP8 was tested independently; NVFP4 requires a supported library recipe and operator evidence."})


## 3. Inspect the evidence

A successful float8 PyTorch GEMM proves that path only. NVFP4 remains unmeasured without its library recipe and operator evidence.

### Acceptance and rollback gate

Record compute capability, dtype/API, scaling recipe, operator success, numerical error, timing, and library version separately for FP8, MXFP8, and NVFP4.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Framework-level FP8 was tested independently; NVFP4 requires a supported library recipe and operator evidence.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:01+00:00",
  "lesson": 20,
  "probe": {
    "api": "torch._scaled_mm",
    "fp8_error": {
      "cosine": 0.99928468,
      "mae": 0.96194851,
      "max_abs": 6.1875,
      "rmse": 1.20845497
    },
    "fp8_gemm": "success",
    "fp8_timing": {
      "median_ms": 0.017568,
      "p90_ms": 0.01856,
      "repeats": 15,
      "samples_ms": [
        0.028544,
        0.020288,
        0.01856,
        0.017568,
        0.017568,
        0.01824,
        0.017312,
        0.01728,
        0.016928,
        0.0176,
        0.017504,
        0.016864,
        0.017824,
        0.017216,
  

## 4. Explain the result

Publish a format-by-hardware-by-library matrix, not a single `supported` checkbox.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).